In [1]:
#login to huggingface

from huggingface_hub import login
login()

In [2]:
import requests
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torchvision

# Load model and processor
model_id = "IDEA-Research/grounding-dino-tiny"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

2025-07-09 16:04:18.320143: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752077058.344321     273 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752077058.351663     273 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


cuda


In [3]:
#download trashcan aircraft dataset
!pip install --upgrade datasets fsspec huggingface_hub

from datasets import load_dataset

# Login using e.g. `huggingface-cli login` to access this dataset
trash_ds = load_dataset("anyaeross/trashcan1")

Resolving data files:   0%|          | 0/4990 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/650 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/1575 [00:00<?, ?it/s]

In [4]:
print(trash_ds)

DatasetDict({
    train: Dataset({
        features: ['image', 'label', 'x_min', 'y_min', 'x_max', 'y_max'],
        num_rows: 8650
    })
    validation: Dataset({
        features: ['image', 'label', 'x_min', 'y_min', 'x_max', 'y_max'],
        num_rows: 1850
    })
    test: Dataset({
        features: ['image', 'label', 'x_min', 'y_min', 'x_max', 'y_max'],
        num_rows: 1836
    })
})


In [8]:
import requests
import torch
from PIL import Image
import matplotlib.pyplot as plt
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
import torchvision
import json

# Load model & processor
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForZeroShotObjectDetection.from_pretrained(model_id).to(device)

subcategories = {
    "trash": [
        'trash_plastic', 'trash_paper', 'trash_fishing_gear',
        'trash_rubber', 'trash_fabric', 'trash_etc', 'trash_wood', 'trash_metal'
    ],
    "animal": [
        'animal_starfish', 'animal_etc', 'animal_shells',
        'animal_eel', 'animal_crab', 'animal_fish'
    ],
    "plant": ['plant'],
    "vehicle": ['rov']
}

categories = ['trash', 'animal', 'plant', 'vehicle']

results_coco = []
category_id_mapping = {}
# category_counter = 1
last_image = None

image_id = -1

for i in range(0, len(trash_ds["test"])):

    labels_curr = []
    bbox_curr = []
    
    image = trash_ds["test"][i]["image"]

    if last_image == image:
        continue

    image_id += 1
    file_name = image.filename
    
    last_image = image
    all_detections = []

    # Phase 1 detection
    for label in categories:
        inputs = processor(images=image, text=[label], return_tensors="pt").to(device)

        with torch.no_grad():
            outputs = model(**inputs)

        results = processor.post_process_grounded_object_detection(
            outputs,
            inputs.input_ids,
            box_threshold=0.3,
            text_threshold=0.2,
            target_sizes=[image.size[::-1]]
        )

        for res in results:
            for j, score in enumerate(res["scores"]):
                all_detections.append({
                    "label": label,
                    "score": score.item(),
                    "box": res["boxes"][j].tolist()
                })

    # NMS
    if all_detections:
        boxes = torch.tensor([det["box"] for det in all_detections])
        scores = torch.tensor([det["score"] for det in all_detections])
        keep_indices = torchvision.ops.nms(boxes, scores, iou_threshold=0.3)
        all_detections = [all_detections[k] for k in keep_indices]

    # Visualize and Phase 2
    # fig, ax = plt.subplots()
    # ax.imshow(image)

    for det in all_detections:
        if det["score"] < 0.3:
            continue

        x_min, y_min, x_max, y_max = det["box"]
        width = x_max - x_min
        height = y_max - y_min

        rect = plt.Rectangle((x_min, y_min), width, height, linewidth=2, edgecolor='red', facecolor='none')
        # ax.add_patch(rect)
        # ax.text(x_min, y_min - 5, f"{det['label']}: {det['score']:.2f}", color='red', fontsize=8, backgroundcolor='white')

        parent_label = det["label"]
        if parent_label not in subcategories:
            continue

        try:
            cropped_image = image.crop((int(x_min), int(y_min), int(x_max), int(y_max)))
            finer_labels = subcategories[parent_label]

            inputs_sub = processor(images=cropped_image, text=finer_labels, return_tensors="pt").to(device)

            with torch.no_grad():
                outputs_sub = model(**inputs_sub)

            results_sub = processor.post_process_grounded_object_detection(
                outputs_sub,
                inputs_sub.input_ids,
                box_threshold=0.3,
                text_threshold=0.2,
                target_sizes=[cropped_image.size[::-1]]
            )

            best_label = None
            best_score = -float('inf')

            for res_sub in results_sub:
                sub_boxes = res_sub["boxes"]
                sub_scores = res_sub["scores"]
                sub_labels = res_sub["labels"]
                text_labels = res_sub.get("text_labels", sub_labels)

                for idx in range(len(sub_boxes)):
                    if sub_scores[idx] > best_score:
                        best_score = sub_scores[idx].item()
                        best_label = text_labels[idx]

            if best_label is None:
                try:
                    logits = outputs_sub.logits.softmax(dim=-1).squeeze(0)
                    avg_logits = logits.mean(dim=0)
                    top_label_idx = avg_logits.argmax().item()
                    best_label = finer_labels[top_label_idx]
                    best_score = avg_logits[top_label_idx].item()
                except Exception as e:
                    print(f"[Warning] Fallback logit-based label selection failed: {e}")
                    best_label = finer_labels[0]
                    best_score = 0.0

            # if best_label not in category_id_mapping:
            #     category_id_mapping[best_label] = category_counter
            #     category_counter += 1
            # category_id = category_id_mapping[best_label]

            labels_curr.append(best_label)
            bbox_curr.append([x_min, y_min, x_max, y_max])

            print(f"Second-stage: {best_label}, bbox: [{x_min:.1f}, {y_min:.1f}, {x_max:.1f}, {y_max:.1f}], score: {best_score:.2f}")

        except Exception as e:
            print(f"[Error] Skipping second-stage detection for box due to: {e}")

    results_coco.append({
        "id": image_id,
        "original_id" : file_name,
        "label" : labels_curr,
        "boxes" : bbox_curr
    })

    # plt.axis("off")
    # plt.show()

    if i % 10 == 0:
        with open("second_stage_detections_coco.json", "w") as f:
            json.dump(results_coco, f, indent=2)



    # plt.close(fig)
    del image, inputs, outputs, results
    import gc
    gc.collect()


# Save output
with open("second_stage_detections_coco.json", "w") as f:
    json.dump(results_coco, f, indent=2)

print(f"Saved {len(results_coco)} second-stage detections to second_stage_detections_coco.json")

Second-stage: trash_metal, bbox: [0.8, 0.0, 479.2, 269.9], score: 0.05
Second-stage: trash_metal, bbox: [0.5, 208.0, 132.6, 269.5], score: 0.05
Second-stage: animal _ shells _ eel _ crab _ fish, bbox: [155.8, 100.9, 306.7, 228.5], score: 0.33
Second-stage: trash_metal, bbox: [1.3, 0.8, 478.7, 330.6], score: 0.05
Second-stage: animal _ shells, bbox: [274.3, 178.6, 316.5, 209.0], score: 0.42
Second-stage: trash_metal, bbox: [1.1, 0.5, 479.2, 269.2], score: 0.05
Second-stage: animal _ shells, bbox: [183.7, 7.4, 196.7, 27.1], score: 0.31
Second-stage: trash_metal, bbox: [0.7, 0.4, 479.7, 269.3], score: 0.05
Second-stage: trash_metal, bbox: [242.3, 100.2, 256.4, 112.0], score: 0.05
Second-stage: trash_metal, bbox: [1.1, 0.5, 479.0, 269.2], score: 0.05
[Warning] Fallback logit-based label selection failed: list index out of range
Second-stage: animal_starfish, bbox: [1.5, 82.1, 172.9, 192.7], score: 0.00
[Warning] Fallback logit-based label selection failed: list index out of range
Second-st

The channel dimension is ambiguous. Got image shape (3, 480, 3). Assuming channels are the first dimension. Use the [input_data_format](https://huggingface.co/docs/transformers/main/internal/image_processing_utils#transformers.image_transforms.rescale.input_data_format) parameter to assign the channel dimension.


Second-stage: animal _ shells fish, bbox: [254.5, 172.1, 296.3, 211.1], score: 0.38
[Error] Skipping second-stage detection for box due to: selected index k out of range
[Warning] Fallback logit-based label selection failed: list index out of range
Second-stage: animal_starfish, bbox: [341.4, 206.9, 478.9, 269.3], score: 0.00
Second-stage: animal _ shells, bbox: [63.1, 49.7, 88.6, 70.4], score: 0.40
Second-stage: trash_metal, bbox: [0.1, -0.1, 480.1, 359.9], score: 0.06
Second-stage: animal _ shells fish, bbox: [302.8, 0.4, 379.2, 24.8], score: 0.44
Second-stage: animal _ shells, bbox: [383.2, 3.0, 479.8, 152.5], score: 0.31
Second-stage: animal _ shells, bbox: [147.3, 113.5, 233.2, 178.3], score: 0.49
Second-stage: trash _ paper fabric, bbox: [0.8, 0.5, 479.6, 269.3], score: 0.45
Second-stage: animal _ shells _ crab fish, bbox: [172.0, 173.7, 202.0, 209.2], score: 0.42
Second-stage: trash_metal, bbox: [0.8, 0.4, 479.5, 269.3], score: 0.05
Second-stage: animal _ shells, bbox: [180.0, 1

In [9]:
import shutil

# Move file to working directory
shutil.move("second_stage_detections_coco.json", "/kaggle/working/second_stage_detections_coco.json")

'/kaggle/working/second_stage_detections_coco.json'